# Human Activity Recognition - Training Model

## Obiettivo
Creare e valutare modelli CNN per riconoscimento attività umane utilizzando:
- **1 sensore** posizionato al **braccio (wrist)**
- **Sensori**: Accelerometro, Giroscopio, Magnetometro
- **Frequenze**: 25Hz e 13Hz

## Configurazioni da testare
| Config | Sensori | Frequenza | Input Shape | Canali |
|--------|---------|-----------|-------------|--------|
| A | Acc + Gyro + Mag | 25Hz | (64, 9) | 9 |
| B | Acc + Gyro + Mag | 13Hz | (32, 9) | 9 |
| C | Acc | 25Hz | (64, 3) | 3 |

## Attività da riconoscere
1. Walking (Camminare)
2. Running (Correre)
3. Standing (In piedi)
4. Sitting (Seduto)
5. Upstairs (Salire scale)
6. Downstairs (Scendere scale)

## Note sul Realismo
- Dataset train/test generati separatamente per evitare data leakage
- Variabilità inter-sessione e intra-sessione
- Rumore realistico e drift dei sensori
- Target accuracy: ~70-85%

---
## 1. Setup e Import Librerie

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
import time
import os

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

print(f"TensorFlow version: {tf.__version__}")
print(f"Numpy version: {np.__version__}")

# Seed per riproducibilità
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# Configurazione
ACTIVITIES = ['Walking', 'Running', 'Standing', 'Sitting', 'Upstairs', 'Downstairs']
NUM_CLASSES = len(ACTIVITIES)
ACTIVITY_MAP = {activity: i for i, activity in enumerate(ACTIVITIES)}

# Parametri sensori
ACC_RANGE = 16  # ±16g
GYRO_RANGE = 2000  # ±2000°/s
MAG_RANGE = 4900  # ±4900µT

# Parametri finestre
FREQ_HIGH = 25  # Hz
FREQ_LOW = 13   # Hz
WINDOW_SIZE_HIGH = 64  # samples @ 25Hz = 2.56s
WINDOW_SIZE_LOW = 32   # samples @ 13Hz = ~2.46s
OVERLAP = 0.5  # 50% overlap

# Dataset - generazione separata per train/test
TRAIN_SESSIONS_PER_ACTIVITY = 20  # sessioni di training
TEST_SESSIONS_PER_ACTIVITY = 5    # sessioni di test
SESSION_DURATION = 60  # secondi per sessione

print(f"Attività: {ACTIVITIES}")
print(f"Frequenze: {FREQ_HIGH}Hz ({WINDOW_SIZE_HIGH} samples), {FREQ_LOW}Hz ({WINDOW_SIZE_LOW} samples)")
print(f"Overlap: {OVERLAP*100}%")
print(f"Train sessions per activity: {TRAIN_SESSIONS_PER_ACTIVITY}")
print(f"Test sessions per activity: {TEST_SESSIONS_PER_ACTIVITY}")

---
## 2. Generazione Dati Sintetici Realistici

In [ ]:
def generate_sensor_signal(activity, sensor_type, duration, freq, session_seed=None):
    """
    Genera segnale sintetico REALISTICO per un sensore specifico.
    Ogni sessione ha parametri leggermente diversi per simulare variabilità.
    
    Parameters:
    - activity: nome attività
    - sensor_type: 'acc', 'gyro', 'mag'
    - duration: durata in secondi
    - freq: frequenza di campionamento (Hz)
    - session_seed: seed per variabilità tra sessioni
    
    Returns:
    - signal: array (n_samples, 3) per x,y,z
    """
    if session_seed is not None:
        np.random.seed(session_seed)
    
    n_samples = int(duration * freq)
    t = np.linspace(0, duration, n_samples)
    
    # Parametri BASE per attività
    base_params = {
        'Walking': {
            'acc': {'mean': [0.2, 9.8, 0.3], 'amp': [2.0, 3.0, 1.5], 'freq': [2, 2, 2]},
            'gyro': {'mean': [0, 0, 0], 'amp': [50, 30, 80], 'freq': [2, 2, 2]},
            'mag': {'mean': [20, -10, 40], 'amp': [5, 5, 5], 'freq': [1, 1, 1]}
        },
        'Running': {
            'acc': {'mean': [0.5, 9.8, 0.5], 'amp': [5.0, 7.0, 4.0], 'freq': [3.5, 3.5, 3.5]},
            'gyro': {'mean': [0, 0, 0], 'amp': [150, 100, 200], 'freq': [3.5, 3.5, 3.5]},
            'mag': {'mean': [20, -10, 40], 'amp': [8, 8, 8], 'freq': [1.5, 1.5, 1.5]}
        },
        'Standing': {
            'acc': {'mean': [0, 9.8, 0], 'amp': [0.3, 0.3, 0.3], 'freq': [0.5, 0.5, 0.5]},
            'gyro': {'mean': [0, 0, 0], 'amp': [10, 10, 10], 'freq': [0.3, 0.3, 0.3]},
            'mag': {'mean': [20, -10, 40], 'amp': [3, 3, 3], 'freq': [0.2, 0.2, 0.2]}
        },
        'Sitting': {
            'acc': {'mean': [0, 9.8, 0], 'amp': [0.2, 0.2, 0.2], 'freq': [0.3, 0.3, 0.3]},
            'gyro': {'mean': [0, 0, 0], 'amp': [5, 5, 5], 'freq': [0.2, 0.2, 0.2]},
            'mag': {'mean': [20, -10, 40], 'amp': [2, 2, 2], 'freq': [0.1, 0.1, 0.1]}
        },
        'Upstairs': {
            'acc': {'mean': [0.3, 10.5, 0.2], 'amp': [3.0, 4.5, 2.0], 'freq': [1.3, 1.3, 1.3]},
            'gyro': {'mean': [0, 0, 0], 'amp': [100, 80, 120], 'freq': [1.3, 1.3, 1.3]},
            'mag': {'mean': [20, -10, 40], 'amp': [6, 6, 6], 'freq': [1, 1, 1]}
        },
        'Downstairs': {
            'acc': {'mean': [0.2, 9.2, 0.3], 'amp': [3.5, 5.0, 2.5], 'freq': [1.4, 1.4, 1.4]},
            'gyro': {'mean': [0, 0, 0], 'amp': [120, 90, 140], 'freq': [1.4, 1.4, 1.4]},
            'mag': {'mean': [20, -10, 40], 'amp': [7, 7, 7], 'freq': [1, 1, 1]}
        }
    }
    
    p = base_params[activity][sensor_type]
    
    # VARIABILITÀ INTER-SESSIONE: ogni sessione ha parametri leggermente diversi
    mean_variation = [np.random.normal(0, 0.3) for _ in range(3)]
    amp_variation = [np.random.uniform(0.7, 1.3) for _ in range(3)]  # ±30% variazione
    freq_variation = [np.random.uniform(0.85, 1.15) for _ in range(3)]  # ±15% variazione
    
    signal_data = np.zeros((n_samples, 3))
    
    for axis in range(3):
        mean_val = p['mean'][axis] + mean_variation[axis]
        amp_val = p['amp'][axis] * amp_variation[axis]
        freq_val = p['freq'][axis] * freq_variation[axis]
        
        # Componente principale (sinusoide con fase casuale)
        phase = np.random.uniform(0, 2*np.pi)
        main_component = mean_val + amp_val * np.sin(2 * np.pi * freq_val * t + phase)
        
        # Componenti armoniche con variabilità
        harmonic1 = (amp_val * np.random.uniform(0.2, 0.4)) * np.sin(2 * np.pi * freq_val * 2 * t + np.random.uniform(0, 2*np.pi))
        harmonic2 = (amp_val * np.random.uniform(0.1, 0.2)) * np.sin(2 * np.pi * freq_val * 3 * t + np.random.uniform(0, 2*np.pi))
        
        # RUMORE REALISTICO (più alto)
        noise_level = amp_val * 0.35  # 35% di rumore invece di 10%
        noise = np.random.normal(0, noise_level, n_samples)
        
        # DRIFT lento del sensore
        drift = np.random.uniform(-0.5, 0.5) * np.linspace(0, 1, n_samples)
        
        # SPIKE casuali (outliers)
        n_spikes = np.random.randint(0, 5)
        spike_signal = np.zeros(n_samples)
        for _ in range(n_spikes):
            spike_pos = np.random.randint(0, n_samples)
            spike_signal[spike_pos] = np.random.normal(0, amp_val * 2)
        
        # VARIABILITÀ INTRA-SESSIONE: velocità che cambia nel tempo
        speed_modulation = 1 + 0.2 * np.sin(2 * np.pi * 0.1 * t)  # variazione lenta
        
        signal_data[:, axis] = (main_component + harmonic1 + harmonic2) * speed_modulation + noise + drift + spike_signal
    
    return signal_data

In [ ]:
def create_windows(data, window_size, overlap=0.5):
    """
    Crea finestre sliding da segnale continuo.
    """
    step = int(window_size * (1 - overlap))
    n_windows = (len(data) - window_size) // step + 1
    
    windows = []
    for i in range(n_windows):
        start = i * step
        end = start + window_size
        if end <= len(data):
            windows.append(data[start:end])
    
    return np.array(windows)

In [ ]:
def generate_dataset_split(freq, window_size, n_sessions_per_activity, split_name='train', base_seed=42):
    """
    Genera dataset con sessioni separate per train/test.
    
    IMPORTANTE: Train e Test sono generati con sessioni DIVERSE per evitare data leakage!
    
    Parameters:
    - split_name: 'train' o 'test'
    - base_seed: seed base per generazione
    
    Returns:
    - X_acc, X_gyro, X_mag, y
    """
    X_acc_list, X_gyro_list, X_mag_list, y_list = [], [], [], []
    
    for activity_idx, activity in enumerate(ACTIVITIES):
        print(f"Generando {split_name} data per {activity} @ {freq}Hz...")
        
        for session_id in range(n_sessions_per_activity):
            # Seed unico per ogni sessione
            session_seed = base_seed + activity_idx * 1000 + session_id
            if split_name == 'test':
                session_seed += 10000  # Test ha seed diversi da train
            
            # Genera segnali per questa sessione
            acc_signal = generate_sensor_signal(activity, 'acc', SESSION_DURATION, freq, session_seed)
            gyro_signal = generate_sensor_signal(activity, 'gyro', SESSION_DURATION, freq, session_seed + 1)
            mag_signal = generate_sensor_signal(activity, 'mag', SESSION_DURATION, freq, session_seed + 2)
            
            # Crea finestre da questa sessione
            acc_windows = create_windows(acc_signal, window_size, OVERLAP)
            gyro_windows = create_windows(gyro_signal, window_size, OVERLAP)
            mag_windows = create_windows(mag_signal, window_size, OVERLAP)
            
            n_windows = len(acc_windows)
            
            X_acc_list.append(acc_windows)
            X_gyro_list.append(gyro_windows)
            X_mag_list.append(mag_windows)
            y_list.append(np.full(n_windows, ACTIVITY_MAP[activity]))
    
    X_acc = np.vstack(X_acc_list)
    X_gyro = np.vstack(X_gyro_list)
    X_mag = np.vstack(X_mag_list)
    y = np.concatenate(y_list)
    
    print(f"\n{split_name.upper()} Dataset generato:")
    print(f"  X_acc shape: {X_acc.shape}")
    print(f"  X_gyro shape: {X_gyro.shape}")
    print(f"  X_mag shape: {X_mag.shape}")
    print(f"  y shape: {y.shape}")
    print(f"  Distribuzione classi: {np.bincount(y)}")
    
    return X_acc, X_gyro, X_mag, y

In [ ]:
# Genera dataset TRAIN e TEST separati per 25Hz
print("=" * 80)
print("GENERAZIONE DATASET 25Hz - TRAIN")
print("=" * 80)
X_acc_25_train, X_gyro_25_train, X_mag_25_train, y_25_train = generate_dataset_split(
    FREQ_HIGH, WINDOW_SIZE_HIGH, TRAIN_SESSIONS_PER_ACTIVITY, 'train', base_seed=42
)

print("\n" + "=" * 80)
print("GENERAZIONE DATASET 25Hz - TEST")
print("=" * 80)
X_acc_25_test, X_gyro_25_test, X_mag_25_test, y_25_test = generate_dataset_split(
    FREQ_HIGH, WINDOW_SIZE_HIGH, TEST_SESSIONS_PER_ACTIVITY, 'test', base_seed=42
)

In [ ]:
# Genera dataset TRAIN e TEST separati per 13Hz
print("\n" + "=" * 80)
print("GENERAZIONE DATASET 13Hz - TRAIN")
print("=" * 80)
X_acc_13_train, X_gyro_13_train, X_mag_13_train, y_13_train = generate_dataset_split(
    FREQ_LOW, WINDOW_SIZE_LOW, TRAIN_SESSIONS_PER_ACTIVITY, 'train', base_seed=100
)

print("\n" + "=" * 80)
print("GENERAZIONE DATASET 13Hz - TEST")
print("=" * 80)
X_acc_13_test, X_gyro_13_test, X_mag_13_test, y_13_test = generate_dataset_split(
    FREQ_LOW, WINDOW_SIZE_LOW, TEST_SESSIONS_PER_ACTIVITY, 'test', base_seed=100
)

---
## 3. Visualizzazione Dati Sintetici

In [ ]:
# Visualizza esempi di segnali per ogni attività
fig, axes = plt.subplots(NUM_CLASSES, 3, figsize=(15, 12))
fig.suptitle('Esempi Segnali Sintetici - Accelerometro @ 25Hz', fontsize=16)

for i, activity in enumerate(ACTIVITIES):
    # Trova prima finestra di questa attività
    idx = np.where(y_25_train == i)[0][0]
    sample = X_acc_25_train[idx]
    
    axes[i, 0].plot(sample[:, 0])
    axes[i, 0].set_ylabel(activity)
    axes[i, 0].set_title('Acc X' if i == 0 else '')
    axes[i, 0].grid(True, alpha=0.3)
    
    axes[i, 1].plot(sample[:, 1])
    axes[i, 1].set_title('Acc Y' if i == 0 else '')
    axes[i, 1].grid(True, alpha=0.3)
    
    axes[i, 2].plot(sample[:, 2])
    axes[i, 2].set_title('Acc Z' if i == 0 else '')
    axes[i, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Preprocessing e Normalizzazione

In [ ]:
def normalize_data(X_train, X_test):
    """
    Normalizza dati usando media e std del training set.
    """
    mean = X_train.mean(axis=(0, 1), keepdims=True)
    std = X_train.std(axis=(0, 1), keepdims=True)
    
    X_train_norm = (X_train - mean) / (std + 1e-8)
    X_test_norm = (X_test - mean) / (std + 1e-8)
    
    return X_train_norm, X_test_norm, mean, std

In [ ]:
def prepare_data_for_config(X_acc_train, X_gyro_train, X_mag_train, X_acc_test, X_gyro_test, X_mag_test, sensors=['acc', 'gyro', 'mag']):
    """
    Prepara dati concatenando i sensori richiesti per train e test.
    
    Parameters:
    - sensors: lista di sensori da includere
    
    Returns:
    - X_train, X_test: dati concatenati
    """
    train_list = []
    test_list = []
    
    if 'acc' in sensors:
        train_list.append(X_acc_train)
        test_list.append(X_acc_test)
    if 'gyro' in sensors:
        train_list.append(X_gyro_train)
        test_list.append(X_gyro_test)
    if 'mag' in sensors:
        train_list.append(X_mag_train)
        test_list.append(X_mag_test)
    
    X_train = np.concatenate(train_list, axis=2)
    X_test = np.concatenate(test_list, axis=2)
    
    return X_train, X_test

---
## 5. Definizione Architettura CNN

In [ ]:
def create_cnn_model(input_shape, num_classes=6):
    """
    Crea modello CNN per HAR.
    
    Parameters:
    - input_shape: (timesteps, channels)
    - num_classes: numero di attività
    
    Returns:
    - model: modello Keras compilato
    """
    model = models.Sequential([
        # Primo blocco Conv
        layers.Conv1D(64, kernel_size=5, activation='relu', input_shape=input_shape, padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(0.3),
        
        # Secondo blocco Conv
        layers.Conv1D(128, kernel_size=5, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(0.4),
        
        # Terzo blocco Conv
        layers.Conv1D(256, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling1D(),
        
        # Dense layers
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

---
## 6. Training per le 3 Configurazioni

In [ ]:
# Configurazioni da testare (solo 3)
configs = [
    {
        'name': 'A_AccGyroMag_25Hz',
        'sensors': ['acc', 'gyro', 'mag'],
        'freq': 25,
        'X_acc_train': X_acc_25_train, 'X_gyro_train': X_gyro_25_train, 'X_mag_train': X_mag_25_train,
        'X_acc_test': X_acc_25_test, 'X_gyro_test': X_gyro_25_test, 'X_mag_test': X_mag_25_test,
        'y_train': y_25_train, 'y_test': y_25_test
    },
    {
        'name': 'B_AccGyroMag_13Hz',
        'sensors': ['acc', 'gyro', 'mag'],
        'freq': 13,
        'X_acc_train': X_acc_13_train, 'X_gyro_train': X_gyro_13_train, 'X_mag_train': X_mag_13_train,
        'X_acc_test': X_acc_13_test, 'X_gyro_test': X_gyro_13_test, 'X_mag_test': X_mag_13_test,
        'y_train': y_13_train, 'y_test': y_13_test
    },
    {
        'name': 'C_Acc_25Hz',
        'sensors': ['acc'],
        'freq': 25,
        'X_acc_train': X_acc_25_train, 'X_gyro_train': X_gyro_25_train, 'X_mag_train': X_mag_25_train,
        'X_acc_test': X_acc_25_test, 'X_gyro_test': X_gyro_25_test, 'X_mag_test': X_mag_25_test,
        'y_train': y_25_train, 'y_test': y_25_test
    },
]

# Dizionario per salvare risultati
results = {}

In [ ]:
# Training loop per le 3 configurazioni
EPOCHS = 50
BATCH_SIZE = 64

for config in configs:
    print("\n" + "="*80)
    print(f"CONFIGURAZIONE: {config['name']}")
    print(f"Sensori: {config['sensors']}, Frequenza: {config['freq']}Hz")
    print("="*80)
    
    # Prepara dati (train e test già separati!)
    X_train, X_test = prepare_data_for_config(
        config['X_acc_train'], config['X_gyro_train'], config['X_mag_train'],
        config['X_acc_test'], config['X_gyro_test'], config['X_mag_test'],
        config['sensors']
    )
    y_train = config['y_train']
    y_test = config['y_test']
    
    print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
    
    # Normalizza
    X_train_norm, X_test_norm, mean, std = normalize_data(X_train, X_test)
    
    # One-hot encoding
    y_train_cat = to_categorical(y_train, NUM_CLASSES)
    y_test_cat = to_categorical(y_test, NUM_CLASSES)
    
    # Crea modello
    input_shape = (X_train_norm.shape[1], X_train_norm.shape[2])
    model = create_cnn_model(input_shape, NUM_CLASSES)
    
    print(f"\nModello creato con input shape: {input_shape}")
    model.summary()
    
    # Callbacks
    early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
    
    # Training
    print(f"\nInizio training...")
    start_time = time.time()
    
    history = model.fit(
        X_train_norm, y_train_cat,
        validation_split=0.15,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )
    
    training_time = time.time() - start_time
    
    # Valutazione su TEST SET
    print(f"\nValutazione su test set...")
    start_time = time.time()
    y_pred = model.predict(X_test_norm)
    inference_time = (time.time() - start_time) / len(X_test_norm) * 1000  # ms per sample
    
    y_pred_classes = np.argmax(y_pred, axis=1)
    
    # Metriche
    accuracy = accuracy_score(y_test, y_pred_classes)
    f1 = f1_score(y_test, y_pred_classes, average='weighted')
    
    print(f"\nRisultati:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  F1-Score (weighted): {f1:.4f}")
    print(f"  Training time: {training_time:.2f}s")
    print(f"  Inference time: {inference_time:.4f}ms/sample")
    
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred_classes, target_names=ACTIVITIES))
    
    # Salva risultati
    results[config['name']] = {
        'model': model,
        'history': history.history,
        'accuracy': accuracy,
        'f1_score': f1,
        'training_time': training_time,
        'inference_time': inference_time,
        'confusion_matrix': confusion_matrix(y_test, y_pred_classes),
        'classification_report': classification_report(y_test, y_pred_classes, target_names=ACTIVITIES, output_dict=True),
        'input_shape': input_shape,
        'mean': mean,
        'std': std
    }
    
    print(f"\n✓ Configurazione {config['name']} completata!")

---
## 7. Visualizzazione Training History

In [ ]:
# Plot training history per tutte le configurazioni
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training History - Tutte le Configurazioni', fontsize=16)

for idx, (name, result) in enumerate(results.items()):
    ax = axes[idx]
    history = result['history']
    
    ax.plot(history['accuracy'], label='Train Accuracy', linewidth=2)
    ax.plot(history['val_accuracy'], label='Val Accuracy', linewidth=2)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 8. Confusion Matrix

In [ ]:
# Plot confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Confusion Matrices - Tutte le Configurazioni', fontsize=16)

for idx, (name, result) in enumerate(results.items()):
    ax = axes[idx]
    cm = result['confusion_matrix']
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=[a[:3] for a in ACTIVITIES],
                yticklabels=[a[:3] for a in ACTIVITIES])
    ax.set_title(f"{name}\nAcc: {result['accuracy']:.3f}", fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.tight_layout()
plt.show()

---
## 9. Conversione a TensorFlow Lite

In [ ]:
def convert_to_tflite(model, model_name, quantize=False):
    """
    Converte modello Keras a TFLite.
    """
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    
    if quantize:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        model_name = model_name.replace('.tflite', '_quantized.tflite')
    
    tflite_model = converter.convert()
    
    # Salva
    output_path = f'saved_model/{model_name}'
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'wb') as f:
        f.write(tflite_model)
    
    size_kb = len(tflite_model) / 1024
    print(f"✓ Modello salvato: {output_path} ({size_kb:.2f} KB)")
    
    return len(tflite_model)

In [ ]:
# Converti tutti i modelli a TFLite
print("=" * 80)
print("CONVERSIONE A TENSORFLOW LITE")
print("=" * 80)

for name, result in results.items():
    print(f"\nConversione {name}...")
    
    # Versione float32
    size_float = convert_to_tflite(result['model'], f'{name}.tflite', quantize=False)
    
    # Versione quantizzata
    size_quant = convert_to_tflite(result['model'], f'{name}.tflite', quantize=True)
    
    results[name]['tflite_size_float'] = size_float
    results[name]['tflite_size_quant'] = size_quant
    
    print(f"  Float32: {size_float/1024:.2f} KB")
    print(f"  Quantized: {size_quant/1024:.2f} KB")
    print(f"  Riduzione: {(1 - size_quant/size_float)*100:.1f}%")

---
## 10. Test TFLite Inference Performance

In [ ]:
def test_tflite_inference(tflite_path, X_test, n_runs=100):
    """
    Testa performance inferenza TFLite.
    """
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    times = []
    for i in range(min(n_runs, len(X_test))):
        input_data = X_test[i:i+1].astype(np.float32)
        
        start = time.time()
        interpreter.set_tensor(input_details[0]['index'], input_data)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]['index'])
        times.append((time.time() - start) * 1000)
    
    return np.mean(times), np.std(times)

In [ ]:
# Test inferenza TFLite
print("=" * 80)
print("TEST INFERENZA TFLITE")
print("=" * 80)

for config in configs:
    name = config['name']
    
    # Prepara test data normalizzato
    X_train, X_test = prepare_data_for_config(
        config['X_acc_train'], config['X_gyro_train'], config['X_mag_train'],
        config['X_acc_test'], config['X_gyro_test'], config['X_mag_test'],
        config['sensors']
    )
    X_train_norm, X_test_norm, _, _ = normalize_data(X_train, X_test)
    
    # Test float32
    tflite_path_float = f'saved_model/{name}.tflite'
    if os.path.exists(tflite_path_float):
        avg_time_float, std_time_float = test_tflite_inference(tflite_path_float, X_test_norm)
        results[name]['tflite_inference_time_float'] = avg_time_float
        print(f"\n{name} (Float32):")
        print(f"  Avg inference time: {avg_time_float:.4f} ± {std_time_float:.4f} ms")
    
    # Test quantized
    tflite_path_quant = f'saved_model/{name}_quantized.tflite'
    if os.path.exists(tflite_path_quant):
        avg_time_quant, std_time_quant = test_tflite_inference(tflite_path_quant, X_test_norm)
        results[name]['tflite_inference_time_quant'] = avg_time_quant
        print(f"  Avg inference time (Quantized): {avg_time_quant:.4f} ± {std_time_quant:.4f} ms")
        print(f"  Speedup: {avg_time_float/avg_time_quant:.2f}x")

---
## 11. Analisi Comparativa

In [ ]:
# Tabella comparativa
comparison_data = []

for name, result in results.items():
    comparison_data.append({
        'Configurazione': name,
        'Accuracy': f"{result['accuracy']:.4f}",
        'F1-Score': f"{result['f1_score']:.4f}",
        'Training Time (s)': f"{result['training_time']:.2f}",
        'Keras Inference (ms)': f"{result['inference_time']:.4f}",
        'TFLite Float (ms)': f"{result.get('tflite_inference_time_float', 0):.4f}",
        'TFLite Quant (ms)': f"{result.get('tflite_inference_time_quant', 0):.4f}",
        'Model Size Float (KB)': f"{result['tflite_size_float']/1024:.2f}",
        'Model Size Quant (KB)': f"{result['tflite_size_quant']/1024:.2f}",
    })

df_comparison = pd.DataFrame(comparison_data)
print("\n" + "=" * 120)
print("TABELLA COMPARATIVA - TUTTE LE CONFIGURAZIONI")
print("=" * 120)
print(df_comparison.to_string(index=False))

# Salva CSV
df_comparison.to_csv('../results/model_comparison.csv', index=False)
print("\n✓ Tabella salvata in: ../results/model_comparison.csv")

---
## 12. Grafici Comparativi

In [ ]:
# Grafico: Accuracy e Inference Time
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

config_names = list(results.keys())
accuracies = [results[name]['accuracy'] for name in config_names]
inference_times = [results[name].get('tflite_inference_time_quant', 0) for name in config_names]
colors = ['#2ecc71', '#e74c3c', '#3498db']

# Accuracy
bars1 = ax1.bar(range(len(config_names)), accuracies, color=colors, alpha=0.8, edgecolor='black')
ax1.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax1.set_title('Confronto Accuracy', fontsize=14, fontweight='bold')
ax1.set_xticks(range(len(config_names)))
ax1.set_xticklabels(config_names, rotation=15, ha='right')
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim([0, 1])

for bar, acc in zip(bars1, accuracies):
    ax1.text(bar.get_x() + bar.get_width()/2, acc + 0.02, f'{acc:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

# Inference time
bars2 = ax2.bar(range(len(config_names)), inference_times, color=colors, alpha=0.8, edgecolor='black')
ax2.set_ylabel('Tempo Inferenza (ms)', fontsize=12, fontweight='bold')
ax2.set_title('Confronto Tempo Inferenza TFLite (Quantized)', fontsize=14, fontweight='bold')
ax2.set_xticks(range(len(config_names)))
ax2.set_xticklabels(config_names, rotation=15, ha='right')
ax2.grid(axis='y', alpha=0.3)

for bar, time_val in zip(bars2, inference_times):
    ax2.text(bar.get_x() + bar.get_width()/2, time_val + 0.01, f'{time_val:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/comparison.png', dpi=300, bbox_inches='tight')
plt.show()

---
## 13. Conclusioni

In [ ]:
print("=" * 80)
print("ANALISI E RACCOMANDAZIONI")
print("=" * 80)

best_acc_name = max(results, key=lambda x: results[x]['accuracy'])
best_acc = results[best_acc_name]['accuracy']

fastest_name = min(results, key=lambda x: results[x].get('tflite_inference_time_quant', float('inf')))
fastest_time = results[fastest_name].get('tflite_inference_time_quant', 0)

print(f"\n📊 MIGLIORE ACCURACY:")
print(f"  Configurazione: {best_acc_name}")
print(f"  Accuracy: {best_acc:.4f}")
print(f"  F1-Score: {results[best_acc_name]['f1_score']:.4f}")

print(f"\n⚡ PIÙ VELOCE:")
print(f"  Configurazione: {fastest_name}")
print(f"  Tempo inferenza: {fastest_time:.4f} ms")
print(f"  Accuracy: {results[fastest_name]['accuracy']:.4f}")

print(f"\n🎯 CONFRONTO:")
print(f"  A (Acc+Gyro+Mag @ 25Hz): Acc={results['A_AccGyroMag_25Hz']['accuracy']:.3f}")
print(f"  B (Acc+Gyro+Mag @ 13Hz): Acc={results['B_AccGyroMag_13Hz']['accuracy']:.3f}")
print(f"  C (Solo Acc @ 25Hz): Acc={results['C_Acc_25Hz']['accuracy']:.3f}")

freq_impact = results['A_AccGyroMag_25Hz']['accuracy'] - results['B_AccGyroMag_13Hz']['accuracy']
sensor_impact = results['A_AccGyroMag_25Hz']['accuracy'] - results['C_Acc_25Hz']['accuracy']

print(f"\n  Riduzione frequenza (25Hz→13Hz): {freq_impact*100:+.2f}% accuracy")
print(f"  Aggiunta Gyro+Mag: {sensor_impact*100:+.2f}% accuracy")

print("\n" + "=" * 80)

---
## 14. Export Risultati

In [ ]:
import pickle

# Salva risultati (senza modelli Keras)
results_export = {}
for name, result in results.items():
    results_export[name] = {k: v for k, v in result.items() if k != 'model'}

with open('../results/all_results.pkl', 'wb') as f:
    pickle.dump(results_export, f)

print("✓ Risultati salvati in: ../results/all_results.pkl")

print("\n" + "=" * 80)
print("✅ TRAINING COMPLETATO!")
print("=" * 80)
print(f"\nModelli TFLite generati: {len(results) * 2} (float + quantized)")
print(f"Risultati in: ../results/")
print(f"Modelli in: ../models/")
print("\nProssimi passi:")
print("  1. Testare modelli TFLite su computer")
print("  2. Deploy su smartphone Android")
print("  3. Confrontare performance reali")
print("=" * 80)